### Importing CybORG

To use the CybORG environment, it is necessary to import the CybORG class. CybORG stands for **Cyb**er **O**perations **R**esearch **G**ym, so remember to capitalise correctly when importing. 

In [ ]:
from CybORG import CybORG

### Instantiating CybORG

ALthough CybORG uses an OpenAI gym API, it is not run by calling gym.make(). Instead it has to be manually instantiated. The constructor has two mandatory string parameters: a mode-type which specifies which engine will be used under the hood and the path to a .yaml scenario file which defines the network layout and agent action spaces.

The only currently supported mode is simulation, while CAGE challenge 2 uses Scenario 2.

In [ ]:
import inspect

path = str(inspect.getfile(CybORG))
path = path[:-10] + '/Shared/Scenarios/Scenario2.yaml'

env = CybORG(path, 'sim')

### Using an Agent with CybORG

Scenario 1b has multiple actors involved: Red team will be attacking the network, Blue team will be defending the network, while Green team represents noise generated by users. Normally the roles of Red and Green will be handled by internal rules-based agents, while Blue team interacts with the external API. However, for demonstration purposes, it will be easier to first examine a Red agent.

CybORG uses an OpenAI Gym interface to interact with agents. Thus, we can begin the scenario by calling the reset method. It is necessary to specify which team you are on as a string parameter. Without using any wrappers, CybORG will return a results object which contains various bits of data. We can get the observation by accessing the corresponding attribute.

In [ ]:
results = env.reset(agent='Red')
obs = results.observation

print(obs)

We can see that the above observation outputs a messy dictionary. In order to understand raw CybORG observations, please go to the observation tutorial. We will show how to train a neural network-based agent with CybORG down below.

Because of the complexties of Cybersecurity, the action space in CybORG is generated on the fly. For Scenario 1b, this only needs to be extracted at the beginning of the scenario. This can also be found in the results object. It is another messy dictionary, so we will only print out the keys. You can learn more in the action_space tutorial.

In [ ]:
action_space = results.action_space

print(list(action_space.keys()))

Just like OpenAI gym, CybORG uses a step function to input actions and return results. The method itself requires two string parameters: agent is the name of the team that is taking the action and action is the action being performed.

In [ ]:
from CybORG.Agents import B_lineAgent

agent = B_lineAgent()

action = agent.get_action(obs,action_space)

results = env.step(agent='Red',action=action)

The results object contains the new observation, reward and done attributes.

In [ ]:
print(results.observation)
print(76*'-')
print(results.action)
print(76*'-')
print(results.done)

### Adding Opponents

In the example above, only Red Team performed any actions. In order to setup CybORG for the full challenge, we want a Blue Agent to be interacting with the external API while Red and Green take their actions automatically in the background.

We can achieve this by specifying an agents dictionary to pass into CybORG when instantiating the class. Now, whenever the step function is called, the agents will take turn to perform their actions. Note that the turn order is Blue, Green then Red.

In [ ]:
from CybORG.Agents import B_lineAgent, GreenAgent, BlueMonitorAgent

agents = {
    'Red': B_lineAgent,
    'Green': GreenAgent
}

env = CybORG(path,'sim',agents=agents)

results = env.reset(agent='Blue')
obs = results.observation
action_space = results.action_space
agent = BlueMonitorAgent()

for step in range(20):
    action = agent.get_action(obs,action_space=action_space)
    results = env.step(agent='Blue',action=action)
    obs = results.observation
    reward = results.reward
    print(reward)

### Using a Neural Network with CybORG

In order to use a Neural network with CybORG, we will need to import a wrapper. The Challenge Wrapper provides all the functionality needed for the Scenario1b challenge and makes the external api much more in line with OpenAI gym.

In [ ]:
from CybORG.Agents.Wrappers import ChallengeWrapper
cyborg = CybORG(path,'sim',agents=agents)
env = ChallengeWrapper(env=cyborg,agent_name='Blue')

obs = env.reset()

print(obs)

Not only is the observation now a vector, but we can extract the action and observation spaces exactly like a gym environment.

The step function also behaves like that of OpenAI gym. The info parameter contains a dictionary form of the standard results object for debugging purposes.

In [ ]:
action = 0

obs, reward, done, info = env.step(action)

print(obs)
print(76*'-')
print(reward)
print(76*'-')
print(done)
print(76*'-')
print(info)